In [ ]:
# ============================================================
# STEP 1 — MOUNT DRIVE AND CHECK SAVED INPUTS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import torch

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

OPENVOICE_DIR = os.path.join(
    BASE_DIR,
    "openvoice_tau_experiment",
    "tau_0.5"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

OPENVOICE_INPUTS = sorted(
    glob.glob(
        os.path.join(
            OPENVOICE_DIR,
            "openvoice_sample_*_tau_0.5.wav"
        )
    )
)

print("=" * 65)
print("NEW GPU SESSION CHECK")
print("=" * 65)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nKathleen reference exists:", os.path.isfile(TARGET_PATH))
print("OpenVoice tau=0.5 files:", len(OPENVOICE_INPUTS))

for number, path in enumerate(OPENVOICE_INPUTS, start=1):
    print(f"Sample {number}: {path}")

assert torch.cuda.is_available(), (
    "❌ الجلسة ليست GPU؛ اختاري T4 GPU"
)

assert os.path.isfile(TARGET_PATH), (
    "❌ Kathleen reference was not found on Drive"
)

assert len(OPENVOICE_INPUTS) == 3, (
    f"❌ Expected 3 OpenVoice inputs, found {len(OPENVOICE_INPUTS)}"
)

print("\n✅ GPU and all saved inputs are ready")

Mounted at /content/drive
NEW GPU SESSION CHECK
CUDA available: True
GPU: Tesla T4

Kathleen reference exists: True
OpenVoice tau=0.5 files: 3
Sample 1: /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_01_tau_0.5.wav
Sample 2: /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_02_tau_0.5.wav
Sample 3: /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_03_tau_0.5.wav

✅ GPU and all saved inputs are ready


In [ ]:
# ============================================================
# STEP 2 — INSTALL THE FIXED SEED-VC VERSION
# ============================================================

import os
import sys
import subprocess

SEED_REPO = "/content/seed-vc"
SEED_COMMIT = "51383efd921027683c89e5348211d93ff12ac2a8"

# منع Transformers من استيراد TensorFlow
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

# تنزيل Seed-VC
if not os.path.isdir(SEED_REPO):
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Plachtaa/seed-vc.git",
            SEED_REPO
        ],
        check=True
    )

# تثبيت نفس الـcommit المستخدم في التجربة السابقة
subprocess.run(
    ["git", "checkout", SEED_COMMIT],
    cwd=SEED_REPO,
    check=True
)

# لا نعيد تثبيت Torch؛ نستخدم نسخة Colab المزودة بدعم GPU
required_packages = [
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "munch==4.0.0",
    "einops==0.8.0",
    "descript-audio-codec==1.0.0",
    "resemblyzer",
    "soundfile==0.12.1",
    "modelscope==1.18.1",
    "funasr==1.1.5",
    "hydra-core==1.3.2",
    "python-dotenv",
    "pyyaml"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + required_packages,
    check=True
)

installed_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=SEED_REPO,
    text=True
).strip()

print("\n" + "=" * 65)
print("SEED-VC INSTALLATION COMPLETE")
print("=" * 65)
print("Repository:", SEED_REPO)
print("Commit:", installed_commit)
print("Python:", sys.version.split()[0])
print("\n✅ Seed-VC dependencies installed")
print("⚠️ الآن اعملي Runtime → Restart session")


SEED-VC INSTALLATION COMPLETE
Repository: /content/seed-vc
Commit: 51383efd921027683c89e5348211d93ff12ac2a8
Python: 3.13.15

✅ Seed-VC dependencies installed
⚠️ الآن اعملي Runtime → Restart session


In [ ]:
# ============================================================
# STEP 3 — POST-RESTART ENVIRONMENT CHECK
# ============================================================

import os

# يجب ضبطها قبل استيراد Transformers
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

from google.colab import drive
drive.mount("/content/drive")

import sys
import glob
import torch
import numpy as np
import librosa
import transformers

SEED_REPO = "/content/seed-vc"
BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

OPENVOICE_DIR = os.path.join(
    BASE_DIR,
    "openvoice_tau_experiment",
    "tau_0.5"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

OPENVOICE_INPUTS = sorted(
    glob.glob(
        os.path.join(
            OPENVOICE_DIR,
            "openvoice_sample_*_tau_0.5.wav"
        )
    )
)

# إضافة Seed-VC إلى Python path
if SEED_REPO not in sys.path:
    sys.path.insert(0, SEED_REPO)

print("=" * 65)
print("POST-RESTART CHECK")
print("=" * 65)

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)
print("Librosa:", librosa.__version__)
print("Transformers:", transformers.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nTensorFlow enabled in Transformers:",
      transformers.utils.is_tf_available())

print("Seed repository exists:", os.path.isdir(SEED_REPO))
print("Kathleen reference exists:", os.path.isfile(TARGET_PATH))
print("OpenVoice tau=0.5 inputs:", len(OPENVOICE_INPUTS))

for index, path in enumerate(OPENVOICE_INPUTS, start=1):
    print(f"Sample {index}: {path}")

assert torch.cuda.is_available(), "❌ GPU is not available"
assert os.path.isdir(SEED_REPO), "❌ Seed-VC repository is missing"
assert os.path.isfile(TARGET_PATH), "❌ Kathleen reference is missing"
assert len(OPENVOICE_INPUTS) == 3, "❌ Expected exactly 3 inputs"
assert transformers.__version__ == "4.46.3", (
    f"❌ Unexpected Transformers version: {transformers.__version__}"
)

print("\n✅ Seed-VC environment and experiment inputs are ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
POST-RESTART CHECK
Python: 3.13.15
Torch: 2.11.0+cu128
NumPy: 1.26.4
Librosa: 0.10.2
Transformers: 4.46.3

CUDA available: True
GPU: Tesla T4

TensorFlow enabled in Transformers: False
Seed repository exists: True
Kathleen reference exists: True
OpenVoice tau=0.5 inputs: 3
Sample 1: /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_01_tau_0.5.wav
Sample 2: /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_02_tau_0.5.wav
Sample 3: /content/drive/MyDrive/IJEEE_VC_data/openvoice_tau_experiment/tau_0.5/openvoice_sample_03_tau_0.5.wav

✅ Seed-VC environment and experiment inputs are ready


In [ ]:
# ============================================================
# STEP 4 — LOAD AND CACHE SEED-VC MODELS
# ============================================================

import os
import sys
import torch
from types import SimpleNamespace

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

SEED_REPO = "/content/seed-vc"

if SEED_REPO not in sys.path:
    sys.path.insert(0, SEED_REPO)

# Seed-VC يعتمد على بعض المسارات النسبية
os.chdir(SEED_REPO)

import inference as seed_inference

# إعدادات ثابتة لجميع تجارب Seed
seed_model_args = SimpleNamespace(
    source=None,
    target=None,
    output=None,

    diffusion_steps=20,
    length_adjust=1.0,
    inference_cfg_rate=0.7,

    f0_condition=False,
    auto_f0_adjust=False,
    semi_tone_shift=0,

    checkpoint=None,
    config=None,
    fp16=True
)

print("Loading Seed-VC models...")
print("The first load may download Seed, Whisper, CAM++ and BigVGAN.\n")

torch.cuda.empty_cache()

seed_model_bundle = seed_inference.load_models(
    seed_model_args
)

print("\n" + "=" * 65)
print("SEED-VC MODELS READY")
print("=" * 65)

print("✅ Seed-VC models loaded and cached")
print("Loaded components:", len(seed_model_bundle))
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Model bundle stored as: seed_model_bundle")

Loading Seed-VC models...
The first load may download Seed, Whisper, CAM++ and BigVGAN.



DiT_seed_v2_uvit_whisper_small_wavenet_b(…):   0%|          | 0.00/440M [00:00<?, ?B/s]

(…)_mel_seed_uvit_whisper_small_wavenet.yml: 0.00B [00:00, ?B/s]

cfm loaded
length_regulator loaded


campplus_cn_common.bin:   0%|          | 0.00/28.0M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]


SEED-VC MODELS READY
✅ Seed-VC models loaded and cached
Loaded components: 7
CUDA available: True
GPU: Tesla T4
Model bundle stored as: seed_model_bundle


In [ ]:
# ============================================================
# STEP 5 — SEED-VC GPU WARM-UP
# ============================================================

import os
import glob
import random
import numpy as np
import torch

from types import SimpleNamespace

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

OPENVOICE_DIR = os.path.join(
    BASE_DIR,
    "openvoice_tau_experiment",
    "tau_0.5"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

OPENVOICE_INPUTS = sorted(
    glob.glob(
        os.path.join(
            OPENVOICE_DIR,
            "openvoice_sample_*_tau_0.5.wav"
        )
    )
)

WARMUP_DIR = "/content/seed_steps_warmup"
os.makedirs(WARMUP_DIR, exist_ok=True)

assert len(OPENVOICE_INPUTS) == 3
assert os.path.isfile(TARGET_PATH)
assert "seed_model_bundle" in globals()

# منع main() من إعادة تحميل المودلات كل مرة
if "original_seed_load_models" not in globals():
    original_seed_load_models = seed_inference.load_models

seed_inference.load_models = lambda args: seed_model_bundle

warmup_args = SimpleNamespace(
    source=OPENVOICE_INPUTS[0],
    target=TARGET_PATH,
    output=WARMUP_DIR,

    diffusion_steps=20,
    length_adjust=1.0,
    inference_cfg_rate=0.7,

    f0_condition=False,
    auto_f0_adjust=False,
    semi_tone_shift=0,

    checkpoint=None,
    config=None,
    fp16=True
)

print("Running Seed-VC warm-up...\n")

for warmup_run in range(1, 3):

    # Seed ثابت
    random.seed(2026)
    np.random.seed(2026)
    torch.manual_seed(2026)
    torch.cuda.manual_seed_all(2026)

    torch.cuda.synchronize()

    print(f"Warm-up run {warmup_run}/2")

    seed_inference.main(warmup_args)

    torch.cuda.synchronize()

    print(f"✅ Warm-up run {warmup_run} completed\n")

print("=" * 65)
print("✅ SEED-VC WARM-UP COMPLETED")
print("=" * 65)
print("Fixed OpenVoice tau: 0.5")
print("Warm-up diffusion steps: 20")
print("GPU:", torch.cuda.get_device_name(0))

Running Seed-VC warm-up...

Warm-up run 1/2


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
100%|██████████| 20/20 [00:01<00:00, 15.02it/s]


RTF: 0.858260388703686


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


✅ Warm-up run 1 completed

Warm-up run 2/2


100%|██████████| 20/20 [00:00<00:00, 20.14it/s]


RTF: 0.33820969896895864
✅ Warm-up run 2 completed

✅ SEED-VC WARM-UP COMPLETED
Fixed OpenVoice tau: 0.5
Warm-up diffusion steps: 20
GPU: Tesla T4


In [ ]:
# ============================================================
# STEP 6 — CONTROLLED SEED DIFFUSION-STEPS EXPERIMENT
# Fixed OpenVoice tau = 0.5
# ============================================================

import os
import glob
import time
import shutil
import random

import numpy as np
import pandas as pd
import torch

from types import SimpleNamespace

# ------------------------------------------------------------
# Fixed experimental configuration
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

OPENVOICE_DIR = os.path.join(
    BASE_DIR,
    "openvoice_tau_experiment",
    "tau_0.5"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

FINAL_OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "hybrid_seed_steps_experiment"
)

LOCAL_WORK_DIR = "/content/hybrid_seed_steps_work"

os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

OPENVOICE_INPUTS = sorted(
    glob.glob(
        os.path.join(
            OPENVOICE_DIR,
            "openvoice_sample_*_tau_0.5.wav"
        )
    )
)

DIFFUSION_STEPS = [5, 10, 20, 30, 50]

BEST_TAU = 0.5
LENGTH_ADJUST = 1.0
CFG_RATE = 0.7
TIMING_REPEATS = 3

# نفس الـseed للعينة نفسها عبر جميع قيم steps
SAMPLE_SEEDS = {
    1: 202601,
    2: 202602,
    3: 202603
}

# مدد عينات Joe الأصلية
SOURCE_DURATIONS = {
    1: 4.973515,
    2: 5.833515,
    3: 6.893515
}

# أزمنة OpenVoice السابقة عند tau=0.5
OPENVOICE_LATENCIES = {
    1: 0.2935,
    2: 0.2824,
    3: 0.3134
}

assert len(OPENVOICE_INPUTS) == 3
assert os.path.isfile(TARGET_PATH)
assert torch.cuda.is_available()
assert "seed_model_bundle" in globals()

# التأكد من عدم إعادة تحميل المودلات
seed_inference.load_models = lambda args: seed_model_bundle

raw_timing_records = []
sample_records = []

RAW_CSV_PATH = os.path.join(
    FINAL_OUTPUT_DIR,
    "seed_steps_timing_repetitions.csv"
)

SAMPLE_CSV_PATH = os.path.join(
    FINAL_OUTPUT_DIR,
    "seed_steps_sample_metrics.csv"
)

SUMMARY_CSV_PATH = os.path.join(
    FINAL_OUTPUT_DIR,
    "seed_steps_summary.csv"
)

print("=" * 75)
print("SEED DIFFUSION-STEPS EXPERIMENT")
print("=" * 75)
print("Fixed OpenVoice tau:", BEST_TAU)
print("Diffusion steps:", DIFFUSION_STEPS)
print("Samples:", len(OPENVOICE_INPUTS))
print("Timing repetitions:", TIMING_REPEATS)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Main experiment
# ------------------------------------------------------------

for steps in DIFFUSION_STEPS:

    print("\n" + "=" * 75)
    print(f"Running diffusion steps = {steps}")
    print("=" * 75)

    for sample_number, source_path in enumerate(
        OPENVOICE_INPUTS,
        start=1
    ):

        local_sample_dir = os.path.join(
            LOCAL_WORK_DIR,
            f"steps_{steps}",
            f"sample_{sample_number:02d}"
        )

        os.makedirs(local_sample_dir, exist_ok=True)

        run_args = SimpleNamespace(
            source=source_path,
            target=TARGET_PATH,
            output=local_sample_dir,

            diffusion_steps=steps,
            length_adjust=LENGTH_ADJUST,
            inference_cfg_rate=CFG_RATE,

            f0_condition=False,
            auto_f0_adjust=False,
            semi_tone_shift=0,

            checkpoint=None,
            config=None,
            fp16=True
        )

        sample_latencies = []

        for repetition in range(1, TIMING_REPEATS + 1):

            fixed_seed = SAMPLE_SEEDS[sample_number]

            random.seed(fixed_seed)
            np.random.seed(fixed_seed)
            torch.manual_seed(fixed_seed)
            torch.cuda.manual_seed_all(fixed_seed)

            torch.cuda.synchronize()
            start_time = time.perf_counter()

            seed_inference.main(run_args)

            torch.cuda.synchronize()
            seed_latency = time.perf_counter() - start_time

            sample_latencies.append(seed_latency)

            duration = SOURCE_DURATIONS[sample_number]
            seed_rtf = seed_latency / duration

            raw_timing_records.append({
                "tau": BEST_TAU,
                "diffusion_steps": steps,
                "sample": sample_number,
                "repetition": repetition,
                "fixed_seed": fixed_seed,
                "duration_seconds": duration,
                "seed_latency_seconds": seed_latency,
                "seed_rtf": seed_rtf
            })

            # حفظ تدريجي لحماية النتائج إذا انقطعت الجلسة
            pd.DataFrame(raw_timing_records).to_csv(
                RAW_CSV_PATH,
                index=False
            )

            print(
                f"Sample {sample_number} | "
                f"Repeat {repetition}/{TIMING_REPEATS} | "
                f"Latency={seed_latency:.4f}s | "
                f"RTF={seed_rtf:.4f}"
            )

        # متوسط قياسات الزمن الثلاثة
        mean_seed_latency = float(np.mean(sample_latencies))
        std_seed_latency = float(
            np.std(sample_latencies, ddof=1)
        )

        duration = SOURCE_DURATIONS[sample_number]
        openvoice_latency = OPENVOICE_LATENCIES[sample_number]

        hybrid_latency = (
            openvoice_latency + mean_seed_latency
        )

        mean_seed_rtf = mean_seed_latency / duration
        hybrid_rtf = hybrid_latency / duration

        # الملف الناتج عن آخر repetition
        generated_files = sorted(
            glob.glob(
                os.path.join(local_sample_dir, "*.wav")
            ),
            key=os.path.getmtime
        )

        assert generated_files, (
            f"No output generated for steps={steps}, "
            f"sample={sample_number}"
        )

        final_audio_path = os.path.join(
            FINAL_OUTPUT_DIR,
            (
                f"hybrid_sample_{sample_number:02d}_"
                f"tau_0.5_steps_{steps:02d}.wav"
            )
        )

        shutil.copy2(
            generated_files[-1],
            final_audio_path
        )

        sample_records.append({
            "tau": BEST_TAU,
            "diffusion_steps": steps,
            "sample": sample_number,
            "fixed_seed": SAMPLE_SEEDS[sample_number],
            "duration_seconds": duration,
            "openvoice_latency_seconds": openvoice_latency,
            "mean_seed_latency_seconds": mean_seed_latency,
            "std_seed_latency_seconds": std_seed_latency,
            "mean_seed_rtf": mean_seed_rtf,
            "mean_hybrid_latency_seconds": hybrid_latency,
            "mean_hybrid_rtf": hybrid_rtf,
            "output_path": final_audio_path
        })

        pd.DataFrame(sample_records).to_csv(
            SAMPLE_CSV_PATH,
            index=False
        )

        print(
            f"\n✅ Sample {sample_number} completed | "
            f"Mean Seed={mean_seed_latency:.4f}s | "
            f"Hybrid={hybrid_latency:.4f}s | "
            f"Hybrid RTF={hybrid_rtf:.4f}"
        )

# ------------------------------------------------------------
# Final summary across the three samples
# ------------------------------------------------------------

sample_df = pd.DataFrame(sample_records)

summary_df = (
    sample_df
    .groupby("diffusion_steps", as_index=False)
    .agg(
        mean_seed_latency=(
            "mean_seed_latency_seconds",
            "mean"
        ),
        std_seed_latency=(
            "mean_seed_latency_seconds",
            "std"
        ),
        mean_seed_rtf=(
            "mean_seed_rtf",
            "mean"
        ),
        mean_hybrid_latency=(
            "mean_hybrid_latency_seconds",
            "mean"
        ),
        std_hybrid_latency=(
            "mean_hybrid_latency_seconds",
            "std"
        ),
        mean_hybrid_rtf=(
            "mean_hybrid_rtf",
            "mean"
        ),
        std_hybrid_rtf=(
            "mean_hybrid_rtf",
            "std"
        )
    )
)

summary_df.insert(0, "tau", BEST_TAU)

summary_df.to_csv(
    SUMMARY_CSV_PATH,
    index=False
)

print("\n" + "=" * 75)
print("FINAL SEED DIFFUSION-STEPS SUMMARY")
print("=" * 75)

print(
    summary_df.round(4).to_string(index=False)
)

print("\n✅ Audio outputs:", FINAL_OUTPUT_DIR)
print("✅ Raw timing metrics:", RAW_CSV_PATH)
print("✅ Sample metrics:", SAMPLE_CSV_PATH)
print("✅ Summary:", SUMMARY_CSV_PATH)

It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


SEED DIFFUSION-STEPS EXPERIMENT
Fixed OpenVoice tau: 0.5
Diffusion steps: [5, 10, 20, 30, 50]
Samples: 3
Timing repetitions: 3
GPU: Tesla T4

Running diffusion steps = 5


100%|██████████| 5/5 [00:00<00:00, 15.23it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.23934900321101196
Sample 1 | Repeat 1/3 | Latency=1.2093s | RTF=0.2432


100%|██████████| 5/5 [00:00<00:00, 21.98it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.18135024600218388
Sample 1 | Repeat 2/3 | Latency=0.9114s | RTF=0.1833


... (82 outputs left out)

100%|██████████| 50/50 [00:02<00:00, 17.07it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.567865410260542
Sample 3 | Repeat 2/3 | Latency=3.9188s | RTF=0.5685


100%|██████████| 50/50 [00:02<00:00, 16.94it/s]


RTF: 0.5690425920220089
Sample 3 | Repeat 3/3 | Latency=3.9279s | RTF=0.5698

✅ Sample 3 completed | Mean Seed=3.9377s | Hybrid=4.2511s | Hybrid RTF=0.6167

FINAL SEED DIFFUSION-STEPS SUMMARY
 tau  diffusion_steps  mean_seed_latency  std_seed_latency  mean_seed_rtf  mean_hybrid_latency  std_hybrid_latency  mean_hybrid_rtf  std_hybrid_rtf
 0.5                5             1.2834            0.2531         0.2167               1.5799              0.2615           0.2677          0.0071
 0.5               10             1.3946            0.1924         0.2370               1.6910              0.2014           0.2880          0.0128
 0.5               20             1.9893            0.2260         0.3390               2.2857              0.2338           0.3900          0.0244
 0.5               30             2.5926            0.2322         0.4430               2.8890              0.2404           0.4939          0.0395
 0.5               50             3.7061            0.2337         0